In [1]:
# Global imports
import biosteam as bst, thermosteam as tmo, biorefineries as bf, numpy as np, pandas as pd
from biorefineries import cellulosic
from biosteam import main_flowsheet as F, units
from biosteam.units.decorators import cost


In [2]:
# Local imports
from atj_saf.atj_bst.etj_chemicals import *
from atj_saf.atj_bst.atj_bst_tea_saf import ConventionalEthanolTEA
from cellulosic_tea_etj import create_cellulosic_ethanol_tea

In [3]:
bst.settings.CEPCI = 2023

In [24]:
bst.settings.electricity_price = 0.07

In [4]:
h2_gas_chems = create_chemicals()
bst.settings.set_thermo(h2_gas_chems) # Setting thermodynamic property pacakge for the chemicals


In [5]:
h2_required = 660000   # [kg/day]
h2_required_hourly = 660000/24
capacity_factor = 0.8



In [ ]:
# Coal requirement calculations

coal_HHV = 8.39          # [kWh/kg] for bituminous coal from https://www.engineeringtoolbox.com/fuels-higher-calorific-values-d_169.html
coal_HHV_MMBTU = (coal_HHV * 3412.14)/1e6   # [MMBTU/kg coal]
coal_price = 2.94        # [$/mmBTU HHV]
coal_usage = 0.188       # [mmBTU HHV/kg H2]
coal_energy = coal_usage*h2_required_hourly  # [MMBTU HHV/hr]
coal_mass = coal_energy/coal_HHV_MMBTU       # [kg/hr]
coal_kg_per_kg_h2 = coal_mass/h2_required_hourly
coal_price_USD_kg = coal_price*coal_HHV_MMBTU


In [7]:
coal_kg_per_kg_h2

6.567030698835532

In [29]:
# Water requirement calculations

water_req = 7.9    # [gal/kg H2]
water_per_day = water_req*h2_required_hourly  # [gal/hr]
water_req_m3 = water_per_day/264.17           # [m3/hr]
water_price = 0.0048  # [$/gal]
water_price_kg = (water_price*264.172)/(h2_gas_chems['Water'].rho(T = 298.15, P = 101325, phase = 'l'))

In [9]:
# Electricity requirement
electricity = 1.04 # [kWh/kg H2]
electricity_total = electricity * h2_required_hourly

In [11]:
class Gasification_Reactor(bst.Unit):

    _N_ins = 2
    _N_outs = 1


    _ins_size_is_fixed = False

    def _init(self):
        pass
 
    
    def _run(self):
        coal, water = self.ins
        hydrogen, = self.outs
        hydrogen.mix_from(coal)
        hydrogen.imass['Hydrogen'] = coal.imass['Coal']/6.567030698835532
        hydrogen.imass['Coal'] = 0

        

    def _design(self):
        pass
        

In [12]:

@cost('Flow rate', units='kg/hr', cost= 5326971228.86, CE=800.8,
      n=0.6, S=1, kW=electricity_total, BM=1, lifetime=30)
class Gasification_Blackbox(bst.Unit):
    pass

    

In [13]:
coal_in = bst.Stream('Coal_In', Coal = coal_mass, units = 'kg/hr')
water_in = bst.Stream('Water_In', Water = water_req_m3, units = 'm3/hr')
h2 = bst.Stream('Hydrogen_Out', Hydrogen = h2_required_hourly, units = 'kg/hr')


In [14]:
gasification_1 = Gasification_Reactor(ins = (coal_in, water_in))
gasification_1.simulate()

In [18]:
Gasification_Cost = Gasification_Blackbox(ins = gasification_1-0)
Gasification_Cost.simulate()

In [19]:
gasification_sys = bst.main_flowsheet.create_system('gasification_sys')
gasification_sys.simulate()

In [20]:
gasification_sys


System: gasification_sys
ins...
[0] Coal_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Coal  1.81e+05
[1] Water_In  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Water  4.55e+04
outs...
[0] s2  
    phase: 'l', T: 298.15 K, P: 101325 Pa
    flow (kmol/hr): Hydrogen  1.36e+04


In [21]:
gas_tea = create_cellulosic_ethanol_tea(gasification_sys)


In [30]:
F.Coal_In.price = coal_price_USD_kg
F.Water_In.price = water_price_kg

AttributeError: can't set attribute 'FOC'